## EDA - Flights and Weather Data at JFK airport - Analysing Sandy Hurrican 2011 

In [ ]:
## import dependencies

# Pandas for manipulation
import pandas as pd 
# SQLalchemy for connecting with SQL database and writing SQL queries
from sqlalchemy import create_engine, text 
# dotent to deal with credentials 
from dotenv import dotenv_values

In [ ]:
## Connect with database
config = dotenv_values()

db_user   = config['POSTGRES_USER']
db_pw     = config['POSTGRES_PASS']
db_host   = config['POSTGRES_HOST']
db_port   = config['POSTGRES_PORT']
db_db     = config['POSTGRES_DB']
db_schema = config['POSTGRES_SCHEMA']
db        = 'postgresql'

In [ ]:
## DATABASE URL -> PostgreSQL connection URL 
URL = f'{db}://{db_user}:{db_pw}@{db_host}:{db_port}/{db_db}'

In [ ]:
## Create an engine with connection URL 
engine = create_engine(URL, echo=True)

In [ ]:
## Set the schema search path 
with engine.begin() as con:
    result = con.execute(text(f'SET search_path TO {db_schema};'))
result

In [ ]:
## Test Querying the DB 
with engine.begin() as con:
    call = con.execute(text(
        '''
            SELECT * FROM filtered_flights;
        '''
    ))
    data = call.all()

In [ ]:
data
df = pd.DataFrame(data)
type(df)

In [ ]:
## Create a function to query 
def query_to_df(query: str)-> pd.DataFrame:
    '''
    This function takes a PostgreSQL Qurey and returns 
    a dataframe from the queried tabel. 

    NOTE: for this function to run you need Pandas and already a valid connection 
    to your DB with SQLalchemy
    '''
    with engine.begin() as con:
        call = con.execute(text(query))
    data = call.all()
    df = pd.DataFrame(data)

    return df

In [ ]:
## FROM FLIGHTS FROM JFK OR TO 
df_flights_jfk = query_to_df(
    '''
        SELECT
            *
        FROM filtered_flights
        ;
    '''
)

In [ ]:
df_weather_jfk = query_to_df(
    '''
        SELECT
            airport_code, 
            station_id,
            date AS sensor_date,
            date_year,
            month_name,
            date_day,
            avg_temp_c,
            min_temp_c,
            max_temp_c,
            max_snow_mm,
            avg_wind_speed,
            precipitation_mm,
            avg_pressure_hpa
        FROM prep_weather_daily
    '''
)

In [ ]:
print(len(df_flights_jfk))
df_flights_jfk.head()

In [ ]:
print(len(df_weather_jfk))
df_weather_jfk.head()

In [ ]:
print(df_weather_jfk.duplicated().sum())
print(df_flights_jfk.duplicated().sum())

In [ ]:
df_weather_jfk.isna().sum()

In [ ]:
print(len(df_flights_jfk))
df_flights_jfk.isna().sum()

In [ ]:
df_flights_jfk.info()

In [ ]:
# flight_filtered_dates = df_flights_jfk[df_flights_jfk['flight_date'] > '2012-11-01']
import datetime
flight_filtered_dates  = df_flights_jfk[(df_flights_jfk['flight_date'] > datetime.datetime(2012,10,20)) & (df_flights_jfk['flight_date'] < datetime.datetime(2012,11,20))]

## Filter the data for a better readability
weather_filtered_dates  = df_weather_jfk[(df_weather_jfk['sensor_date'] > datetime.date(2012,10,20)) & (df_weather_jfk['sensor_date'] < datetime.date(2012,11,20))]

In [ ]:
df_weather_jfk.head()

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

In [ ]:
cancelled_per_day = flight_filtered_dates.groupby('flight_date')['cancelled'].sum()

fig, ax = plt.subplots(figsize=(20,5))
ax.bar(cancelled_per_day.index, cancelled_per_day.values)
ax.set_xlabel('Flight Date')
ax.set_ylabel('Number of Cancelled Flights')
ax.set_title('Number of Cancelled Flights per Day')
ax.set_xticks(cancelled_per_day.index)
plt.xticks(rotation=45)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Windspeed', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['avg_wind_speed'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Wind Speed KmH', fontsize=12)
ax[1].set_title('Average Wind Speed per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
flight_filtered_dates.head()

In [ ]:
weather_filtered_dates.head()

In [ ]:
sum(cancelled_per_day)

In [ ]:
merged_df = pd.merge(
    cancelled_per_day,
    weather_filtered_dates,
    left_on='flight_date',
    right_on='sensor_date',
    how='inner' 
)

In [ ]:
selected_cols = ['cancelled','avg_temp_c','min_temp_c','max_temp_c','max_snow_mm','avg_wind_speed','precipitation_mm','avg_pressure_hpa']
new_df = merged_df[selected_cols].copy()
new_df

In [ ]:
## 2012-10-21 to 2012-11-19	
correlation_matrix = new_df.corr()
correlation_matrix

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Pressure', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['avg_pressure_hpa'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Average Pressure', fontsize=12)
ax[1].set_title('Average Presure Speed per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Windspeed', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['avg_wind_speed'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Wind Speed KmH', fontsize=12)
ax[1].set_title('Average Wind Speed per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(20,15))
ax = sns.heatmap(new_df.corr())


In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Snow', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_filtered_dates['sensor_date'].sort_values(ascending=True), weather_filtered_dates['max_snow_mm'])
ax[1].set_xticks(weather_filtered_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Snow mm', fontsize=12)
ax[1].set_title('Max Snow per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
import datetime
flight_focused_dates  = df_flights_jfk[(df_flights_jfk['flight_date'] > datetime.datetime(2012,11,2)) & (df_flights_jfk['flight_date'] < datetime.datetime(2012,11,13))]

## Filter the data for a better readability
weather_focused_dates  = df_weather_jfk[(df_weather_jfk['sensor_date'] > datetime.date(2012,11,2)) & (df_weather_jfk['sensor_date'] < datetime.date(2012,11,13))]

cancelled_per_day = flight_focused_dates.groupby('flight_date')['cancelled'].sum()


In [ ]:
fig, ax = plt.subplots(2,1, figsize=(20,15))
plt.suptitle('Canclled Flights VS Snow', fontsize=30)
fig.tight_layout()
plt.subplots_adjust(hspace=.5, wspace=.2, top=.9)

ax[0].bar(cancelled_per_day.index, cancelled_per_day.values)
ax[0].set_xticks(cancelled_per_day.index)
ax[0].set_xlabel('Flight Date')
ax[0].set_ylabel('Number of cancelled flights', fontsize=12)
ax[0].set_title('Number of Cancelled Flights per Day')
ax[0].tick_params(axis='x', rotation=45)

ax[1].plot(weather_focused_dates['sensor_date'].sort_values(ascending=True), weather_focused_dates['avg_wind_speed'])
ax[1].set_xticks(weather_focused_dates['sensor_date'])
ax[1].set_xlabel('Weather Sensor Date')
ax[1].set_ylabel('Snow mm', fontsize=12)
ax[1].set_title('Max Snow per Day')
ax[1].tick_params(axis='x', rotation=45)

In [ ]:
# 25.10.2012
# 10.11.2012

flight_dates_canc_num = df_flights_jfk[(df_flights_jfk['flight_date'] > datetime.datetime(2012,10,25)) & (df_flights_jfk['flight_date'] < datetime.datetime(2012,11,10))]
flight_dates_canc_num

In [ ]:
len(flight_dates_canc_num)
cancelled_flights = flight_dates_canc_num[flight_dates_canc_num['cancelled'] == 1]
sum_cancelled = sum(cancelled_flights['cancelled'])

In [ ]:
len(df_flights_jfk)

In [ ]:
all_flights = len(flight_dates_canc_num)

In [ ]:
percentage_canceled = sum_cancelled/all_flights

In [ ]:
round(percentage_canceled * 100,2)

In [ ]:
sum_cancelled